# §1 Individual (unfiltered) (v12 top-50 raw-Sharpe, V4 combo-agnostic filter) — net of costs

Per-combo metrics and per-combo equity/drawdown curves on the
20% OOS test partition with no ML#2 filter. Sizing: `fixed_dollars_500` (risk $500 per trade, flat).

**Cost model:** every trade is charged `contracts × $5.00` round-trip (≈ $3 retail commission + 2 ticks/side slippage on MNQ at $0.50/tick).

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

REPO = Path.cwd().resolve()
while not (REPO / 'src').exists() and REPO.parent != REPO:
    REPO = REPO.parent
sys.path.insert(0, str(REPO))

from scripts.evaluation._top_perf_common import (
    STARTING_EQUITY, POLICIES,
    apply_sizing, metrics_from_pnl, monte_carlo,
    load_setup,
    plot_indiv_equity, plot_indiv_dd,
    plot_combined_equity, plot_combined_dd,
    plot_ml2_indiv_equity, plot_ml2_indiv_dd,
    plot_ml2_combined_equity, plot_ml2_combined_dd,
    plot_mc_sims, plot_mc_pnl, plot_mc_sharpe, plot_mc_dd,
)

_ctx = load_setup(cost_per_contract_rt=5.0, top_strategies_path=REPO / 'evaluation' / 'top_strategies_v12_raw_sharpe_top50.json', version='v4_no_gcid')
bars            = _ctx['bars']
YEARS_SPAN      = _ctx['years_span']
strategies      = _ctx['strategies']
results_raw     = _ctx['results_raw']
combined_raw    = _ctx['combined_raw']
combos_ml2      = _ctx['combos_ml2']
s4_pnl_by_combo = _ctx['s4_pnl_by_combo']
ml2_portfolio   = _ctx['ml2_portfolio']


Top-K source: top_strategies_v12_raw_sharpe_top50.json


Test partition: 514,563 bars  2024-10-22 05:08:00 -> 2026-04-08 20:20:00
Years span: 1.461  (used to annualize Sharpe)
Applying friction: $5.00/contract RT (commission + slippage).
Loaded 50 strategies.
Loaded results_raw from cache (50 combos).
Combined unfiltered trades: 43,846
Loaded combos_ml2 from cache (50 combos).


ML2 portfolio trade counts: {'fixed_dollars_500': 1446}


In [2]:
rows = []
for r in results_raw:
    if r['trades'].empty:
        for policy in POLICIES:
            rows.append({'combo_id': r['combo_id'], 'policy': policy,
                         **metrics_from_pnl(np.array([]), YEARS_SPAN, policy=policy)})
        continue
    t = r['trades'].sort_values('date', kind='mergesort')
    pnl_base = t['actual_pnl'].to_numpy(dtype=float)
    risk_base = t['dollar_risk'].to_numpy(dtype=float)
    r_mult = np.where(risk_base > 0, pnl_base / risk_base, 0.0)
    for policy in POLICIES:
        pnl = apply_sizing(pnl_base, risk_base, policy)
        rows.append({'combo_id': r['combo_id'], 'policy': policy,
                     **metrics_from_pnl(pnl, YEARS_SPAN, policy=policy, r=r_mult)})
perf1 = pd.DataFrame(rows)
perf1

,combo_id,policy,n_trades,trades_per_year,win_rate,total_pnl_dollars,total_return_pct,sharpe_ratio,max_drawdown_pct,max_drawdown_dollars
0,v11_7872,fixed_dollars_500,889,608.5,0.1350,36543.51,73.09,1.5519,19.21,20398.91
1,v11_23634,fixed_dollars_500,2394,1638.6,0.1316,-1317.71,-2.64,-0.0351,47.13,36136.72
2,v11_15760,fixed_dollars_500,350,239.6,0.2029,9226.40,18.45,0.5343,15.54,10895.94
3,v11_2646,fixed_dollars_500,294,201.2,0.1667,-16668.96,-33.34,-1.5449,40.63,20674.91
4,v11_7877,fixed_dollars_500,1595,1091.7,0.3724,15304.03,30.61,0.6105,21.33,15261.60
5,v11_11149,fixed_dollars_500,286,195.8,0.4545,-10237.54,-20.48,-1.2820,23.98,12453.82
6,v11_14258,fixed_dollars_500,541,370.3,0.2366,27809.70,55.62,1.0121,18.48,15719.52
7,v11_694,fixed_dollars_500,242,165.6,0.4835,-3905.81,-7.81,-0.5220,14.57,7759.11
8,v11_9978,fixed_dollars_500,446,305.3,0.2130,19117.99,38.24,1.3625,10.14,6078.40
9,v11_8640,fixed_dollars_500,969,663.2,0.1940,2018.70,4.04,0.0646,31.26,22978.00


In [3]:
plot_indiv_equity(results_raw, 'fixed_dollars_500')

In [4]:
plot_indiv_dd(results_raw, 'fixed_dollars_500')